## Practise 

### Rychlý test systémů
Vytvoříme jednoduchou aplikaci "Sonar", která reaguje na vstup uživatele (hloubku) a vypisuje bezpečnostní varování.

**Zadání:**

- Vytvoříme nový soubor sonar.py.

- Importujeme streamlit.

- Přidáme nadpis aplikace "Echelon Sonar".

- Vytvoříme posuvník (slider) pro nastavení hloubky od 0 do 11 000 metrů.

- Připravíme jednoduchou podmínku:

    - Pokud je hloubka větší než 8000 m, zobrazí se červená hláška (st.error) "CRITICAL PRESSURE".

    - Jinak se zobrazí zelená hlášku (st.success) "Systems Nominal".

- Aplikaci spustíme v terminálu příkazem streamlit run sonar.py.

In [ ]:
import streamlit as st

# 1. UI Elements
st.title("🌊 Echelon Sonar Control")

# 2. Input Widget
current_depth = st.slider("Target Depth (meters)", min_value=0, max_value=11000, value=500)

# 3. Logic & Output
st.write(f"Current reading: **{current_depth} m**")

if current_depth > 8000:
    st.error("⚠️ WARNING: Crushing pressure detected! Hull integrity at risk.")
elif current_depth > 4000:
    st.warning("Caution: Entering Abyssal Zone.")
else:
    st.success("✅ Systems Nominal. Safe for operation.")

## Homework (Projekt: Mars - Final Dashboard)

**Mise:** Velení na Zemi schválilo nasazení našeho analytického softwaru. Naším úkolem je vytvořit `Ares Mission Control Dashboard`, který integruje data (Pandas), pokročilé vizualizace (Matplotlib) a rychlé přehledy (Streamlit).

0. Vstupní data: Pro účely tohoto úkolu použijeme simulaci, která vychází z dat, jež jsme zpracovávali v předchozích lekcích.

```python
import pandas as pd
import numpy as np

def get_mars_mission_data():
    """Generates simulation data for the mission."""
    dates = pd.date_range(start='2035-01-01', periods=30, freq='D')
    df = pd.DataFrame({
        'date': dates,
        'sol': np.arange(1, 31),
        'avg_temp_c': np.random.uniform(-75, -55, 30),
        'pressure_pa': np.random.uniform(600, 620, 30),
        'radiation_rem': np.random.uniform(0.1, 0.5, 30),
        'power_output_kwh': np.linspace(100, 95, 30) - np.random.uniform(0, 5, 30) # Klesající výkon panelů
    })
    return df
```

### Zadání: Level 1 (The Monolith)
Vytvoříme soubor app.py. Všechna logika bude v tomto jednom souboru.

1. Konfigurace:

    - Nastavíme stránce název "Ares Mission Control" a layout `"wide"`.

    - Přidáme hlavní nadpis a boční panel s obrázkem Marsu.

2. Data & KPI:

    - Načteme data pomocí funkce `get_mars_mission_data()`.

    - Vytvoříme 3 sloupce a zobrazíme metriky pro poslední měřený den: Sol, Teplota, Výkon (Power).

3. Matplotlib Graf (Detailed Analysis):

    - Vytvoříme objektový graf (fig, ax), který zobrazí korelaci mezi `Tlakem (X)` a `Teplotou (Y)`.

    - Použijeme `scatter plot`.

    - Graf zobrazíme ve Streamlitu.

4. Streamlit Native Graf (Quick Overview):

    - Zobrazíme vývoj `Výkonu (Power)` v čase pomocí nativního jednoduchého čárového grafu.


### Zadání: Level 2 (The Architect)
Refaktorujeme aplikaci podle principů modularity. Rozdělíme kód do tří souborů:

***utils.py (Backend):**

- Sem přesuneme funkci `get_mars_mission_data`.

- zajistíme, aby se data nenačítala při každém kliknutí znovu.

**views.py (Frontend):**

- Vytvoříme funkci `display_dashboard(df)`, která bude obsahovat kód pro vykreslení metrik a grafů.

**app.py (Main):**

- Zůstane zde pouze vlastní konfigurace stránky, načtení dat z `utils` a zavolání funkce z `views`.

In [ ]:
# utils.py

import pandas as pd
import numpy as np
import streamlit as st

@st.cache_data
def get_mars_mission_data():
    """
    Generates cached simulation data for the mission.
    """
    dates = pd.date_range(start='2035-01-01', periods=30, freq='D')
    df = pd.DataFrame({
        'date': dates,
        'sol': np.arange(1, 31),
        'avg_temp_c': np.random.uniform(-75, -55, 30),
        'pressure_pa': np.random.uniform(600, 620, 30),
        'radiation_rem': np.random.uniform(0.1, 0.5, 30),
        'power_output_kwh': np.linspace(100, 95, 30) - np.random.uniform(0, 5, 30)
    })
    return df

In [ ]:
# views.py

import streamlit as st
import matplotlib.pyplot as plt

def display_dashboard(df):
    """
    Renders the main dashboard elements.
    """
    # 1. KPI Metrics
    st.subheader("📡 Current Status (Last Sol)")
    col1, col2, col3 = st.columns(3)
    
    last_row = df.iloc[-1]
    
    col1.metric("Current Sol", f"{int(last_row['sol'])}")
    col2.metric("Avg Temperature", f"{last_row['avg_temp_c']:.1f} °C")
    col3.metric("Power Output", f"{last_row['power_output_kwh']:.1f} kWh", delta="-0.2 kWh")
    
    st.divider()

    # 2. Charts Layout (2 columns)
    chart_col1, chart_col2 = st.columns(2)
    
    with chart_col1: # Matplotlib Scatter Plot
        st.subheader("📊 Atmosphere Analysis (Matplotlib)")
        
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(df['pressure_pa'], df['avg_temp_c'], color='red', alpha=0.6)
        ax.set_title("Pressure vs. Temperature")
        ax.set_xlabel("Pressure (Pa)")
        ax.set_ylabel("Temp (°C)")
        ax.grid(True, alpha=0.3)
        # Render plot
        st.pyplot(fig)
        
    with chart_col2: # Native Streamlit Chart
        st.subheader("⚡ Power Trends (Native)")
        st.line_chart(df, x='sol', y='power_output_kwh')

In [ ]:
# app.py

import streamlit as st
from utils import get_mars_mission_data
from views import display_dashboard

# 1. App Configuration
st.set_page_config(
    page_title="Ares Mission Control",
    page_icon="🔴",
    layout="wide"
)

# 2. Sidebar
st.sidebar.image("https://upload.wikimedia.org/wikipedia/commons/thumb/0/02/OSIRIS_Mars_true_color.jpg/800px-OSIRIS_Mars_true_color.jpg", caption="Ares Sector 4")
st.sidebar.title("Mission Options")
st.sidebar.info("System v1.02 Online")

# 3. Main Logic
st.title("🚀 Ares Mission Control")

# Load data (Cached)
df_mars = get_mars_mission_data()

# Display View
display_dashboard(df_mars)